# Exploratory Data Analysis - Mobile Money Fraud DetectionSource dataset: [`CiferAI/Cifer-Fraud-Detection-Dataset-AF`](https://huggingface.co/datasets/CiferAI/Cifer-Fraud-Detection-Dataset-AF)The upstream dataset holds 21M mobile money transactions spread over 14 csv shards.This notebook works on the sampled working dataset produced by `download_dataset.py`.

#### Import Packages and the working dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

In [ ]:
DATA_PATH = "../data/fraud_transactions.csv"

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

## Column dictionary| column | meaning || --- | --- || `step` | hour of the simulated month (1-743) || `type` | transaction type: PAYMENT, TRANSFER, CASH_OUT, CASH_IN, DEBIT || `amount` | transaction amount || `nameOrig` | anonymised sender id || `oldbalanceOrg` | sender balance before the transaction || `newbalanceOrig` | sender balance after the transaction || `nameDest` | anonymised receiver id || `oldbalanceDest` | receiver balance before the transaction || `newbalanceDest` | receiver balance after the transaction || `isFraud` | target, 1 when the transaction is fraudulent || `isFlaggedFraud` | rule based flag raised by the platform |

In [ ]:
df.info()

## Data Cleaning

#### Check Null Values

In [ ]:
features_with_na = [feature for feature in df.columns if df[feature].isnull().sum() >= 1]
for feature in features_with_na:
    print(feature, np.round(df[feature].isnull().mean() * 100, 5), "% missing values")

features_with_na

**Handling Duplicates**

In [ ]:
df.duplicated().sum()

`nameOrig` and `nameDest` are anonymised identifiers generated by the synthesiser.They carry no signal and are dropped before modelling, exactly like `case_id` in the original project.

In [ ]:
print("unique senders  :", df["nameOrig"].nunique())
print("unique receivers:", df["nameDest"].nunique())
print("rows            :", len(df))

## Target distribution

The working dataset keeps every fraud row from the downloaded shards and down-samplesthe legitimate rows to 20 per fraud row. The raw upstream fraud rate is roughly 0.13 percent.

In [ ]:
target_counts = df["isFraud"].value_counts()
print(target_counts)
print((target_counts / len(df) * 100).round(2))

plt.figure(figsize=(5, 4))
sns.countplot(x="isFraud", data=df, palette=["#2a9d8f", "#e76f51"])
plt.title("Target distribution in the working dataset")
plt.xticks([0, 1], ["Legitimate", "Fraud"])
plt.show()

## Numerical and categorical features

In [ ]:
num_features = [feature for feature in df.columns if df[feature].dtype != "O"]
cat_features = [feature for feature in df.columns if df[feature].dtype == "O"]

print("Num of Numerical Features   :", len(num_features), num_features)
print("Num of Categorical Features :", len(cat_features), cat_features)

In [ ]:
discrete_features = [feature for feature in num_features if df[feature].nunique() <= 25]
continuous_features = [feature for feature in num_features if feature not in discrete_features]

print("Num of Discrete Features   :", len(discrete_features), discrete_features)
print("Num of Continuous Features :", len(continuous_features), continuous_features)

## Univariate analysis of the monetary columns

In [ ]:
money_cols = ["amount", "oldbalanceOrg", "newbalanceOrig", "oldbalanceDest", "newbalanceDest"]

plt.figure(figsize=(15, 8))
for i, col in enumerate(money_cols):
    plt.subplot(2, 3, i + 1)
    sns.histplot(x=np.log1p(df[col]), color="indianred", bins=50)
    plt.xlabel(f"log1p({col})")
plt.tight_layout()
plt.show()

In [ ]:
df[money_cols].describe().T

**Skewness of the monetary columns**

In [ ]:
df[money_cols].skew(axis=0, skipna=True)

All monetary columns are strongly right skewed, which is why a Yeo-Johnson`PowerTransformer` is used in the preprocessing pipeline.

## Transaction type

In [ ]:
plt.figure(figsize=(10, 4))
sns.countplot(x="type", data=df, order=df["type"].value_counts().index, palette="viridis")
plt.title("Transaction type distribution")
plt.show()

df["type"].value_counts()

**Fraud rate per transaction type**

In [ ]:
fraud_by_type = (
    df.groupby("type")["isFraud"]
    .agg(transactions="count", frauds="sum")
    .assign(fraud_rate=lambda d: (d["frauds"] / d["transactions"] * 100).round(2))
    .sort_values("fraud_rate", ascending=False)
)
fraud_by_type

In [ ]:
plt.figure(figsize=(10, 4))
sns.barplot(x=fraud_by_type.index, y=fraud_by_type["fraud_rate"], palette="rocket")
plt.ylabel("fraud rate [%]")
plt.title("Fraud rate per transaction type")
plt.show()

In this synthetic dataset the fraud rate is close to identical across all five transactiontypes. In the original PaySim data fraud only ever occurs in `TRANSFER` and `CASH_OUT`,so this is the first sign that the synthesiser did not preserve the target relationship.

## Temporal behaviour

In [ ]:
df["hour_of_day"] = df["step"] % 24
df["day_of_month"] = df["step"] // 24

plt.figure(figsize=(12, 4))
sns.lineplot(
    data=df.groupby("hour_of_day")["isFraud"].mean().mul(100).reset_index(),
    x="hour_of_day",
    y="isFraud",
    marker="o",
    color="#e76f51",
)
plt.ylabel("fraud rate [%]")
plt.title("Fraud rate per hour of day")
plt.show()

## Amount distribution split by the target

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
sns.kdeplot(data=df, x=np.log1p(df["amount"]), hue="isFraud", common_norm=False, fill=True)
plt.xlabel("log1p(amount)")
plt.title("Amount")

plt.subplot(1, 2, 2)
sns.kdeplot(data=df, x=np.log1p(df["oldbalanceOrg"]), hue="isFraud", common_norm=False, fill=True)
plt.xlabel("log1p(oldbalanceOrg)")
plt.title("Sender balance before")
plt.tight_layout()
plt.show()

In [ ]:
df.groupby("isFraud")[money_cols].median().T

## Balance consistency features

In real mobile money logs the balance after a transaction should equal the balance beforeminus the amount. The residual of that identity is one of the strongest fraud signals,so it is engineered explicitly in `FraudFeatureEngineer`.

In [ ]:
df["error_balance_orig"] = df["oldbalanceOrg"] - df["amount"] - df["newbalanceOrig"]
df["error_balance_dest"] = df["oldbalanceDest"] + df["amount"] - df["newbalanceDest"]

df.groupby("isFraud")[["error_balance_orig", "error_balance_dest"]].describe().T

## Correlation with the target

In [ ]:
corr_cols = money_cols + ["step", "hour_of_day", "day_of_month",
                          "error_balance_orig", "error_balance_dest", "isFlaggedFraud", "isFraud"]

plt.figure(figsize=(10, 8))
sns.heatmap(df[corr_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation matrix")
plt.show()

In [ ]:
df[corr_cols].corr()["isFraud"].drop("isFraud").sort_values(key=abs, ascending=False)

## isFlaggedFraud

In [ ]:
pd.crosstab(df["isFlaggedFraud"], df["isFraud"])

`isFlaggedFraud` is a platform side rule, it fires on a handful of rows and would leakplatform logic into the model, so it is listed in `drop_columns` in `config/schema.yaml`.

## Conclusions* No missing values and no duplicated rows.* `nameOrig` / `nameDest` are random identifiers, `isFlaggedFraud` is a leaky platform rule, `step` is replaced by `hour_of_day` and `day_of_month`. All four are dropped.* All monetary columns are heavily right skewed and get a Yeo-Johnson power transform.* `type` is nominal and is one hot encoded.* **The target carries almost no signal in this synthetic dataset.** Every conditional  distribution above is nearly identical for fraud and non fraud rows. This is quantified  in notebook 2.